# 05. Macro Context — 매크로 영향 분리

구매 단가의 변동을 **외부 매크로 변수**(환율, 유가, 의류 원가 지수)로 얼마나 설명할 수 있는지 측정합니다.

### 왜 이 분석이 필요한가

구매가 인상이 회사 내부 협상력 부족인지, **세계 어디에서도 같은 흐름**인지 분리해야 의사결정이 다릅니다.

| 인상 요인 | 합리적 행동 |
|---|---|
| 매크로(환율↑·유가↑) | 헷징, 결제 통화 변경, 지역 다변화 |
| 자사 매입 패턴 | 발주 주기·수량 조정 |
| 시장 구조 | 신규 공급사 발굴, 대체재 검토 |

### 사용 패키지

- `pandas` — merge, rolling correlation
- `numpy` — 회귀 전 표준화
- `statsmodels.api` — OLS (Ordinary Least Squares)로 매크로 → 매입가 영향 추정
- `matplotlib` — 동행 시계열 플롯, 잔차 플롯

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')

## 1. 매입 단가 + 매크로 결합

In [ ]:
purchases = pd.read_csv(DATA_DIR / 'sample_purchases.csv', parse_dates=['date'])
macro = pd.read_csv(DATA_DIR / 'sample_macro.csv', parse_dates=['date'])

TARGET = 'SKU-A001-COTTON'
p = purchases[purchases['product_id'] == TARGET][['date', 'unit_cost_krw']].sort_values('date')
panel = p.merge(macro, on='date', how='inner').set_index('date')
panel.head()

## 2. 동행 시각화

각 변수가 같은 방향으로 움직이는지 우선 눈으로 확인. 단위가 달라 표준화(z-score) 후 비교.

In [ ]:
z = (panel - panel.mean()) / panel.std()
fig, ax = plt.subplots(figsize=(11, 4))
for col in ['unit_cost_krw', 'fx_krw_usd', 'oil_index_usd', 'apparel_cost_index']:
    ax.plot(z.index, z[col], label=col, linewidth=1.5)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title(f'{TARGET} 매입가 vs 매크로 (표준화)')
ax.set_ylabel('z-score')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.3)
fig.autofmt_xdate(); fig.tight_layout()
fig.savefig(DATA_DIR / f'macro_overlay_{TARGET}.png', dpi=120)
plt.show()

## 3. 동시 상관 + 시차 상관

**동시 상관**: 같은 달 매크로가 매입가와 얼마나 같이 움직이나.

**시차(lag) 상관**: 매크로가 N개월 앞서 움직인 뒤 매입가가 따라가나. 양수 lag = 매크로가 선행지표.

In [ ]:
corr = panel.corr()['unit_cost_krw'].drop('unit_cost_krw').round(3)
print('동시 상관 (Pearson):')
print(corr)

lags = range(-3, 4)
lag_table = {}
for var in ['fx_krw_usd', 'oil_index_usd', 'apparel_cost_index']:
    lag_table[var] = [round(panel['unit_cost_krw'].corr(panel[var].shift(L)), 3) for L in lags]
lag_df = pd.DataFrame(lag_table, index=[f'lag={L:+d}' for L in lags])
print('\n시차 상관 (음수 lag = 매크로가 후행):')
lag_df

## 4. OLS — 매크로가 매입가 변동을 얼마나 설명하나

회귀 모형:

$$\Delta \log(\text{cost}_t) = \alpha + \beta_1 \Delta\log(FX_t) + \beta_2 \Delta\log(Oil_t) + \beta_3 \Delta\log(App_t) + \epsilon_t$$

1차 차분 + log를 사용해 비정상성과 단위 문제를 해결합니다. 결정계수($R^2$)가 매크로가 설명하는 비중.

In [ ]:
ret = np.log(panel).diff().dropna()
y = ret['unit_cost_krw']
X = ret[['fx_krw_usd', 'oil_index_usd', 'apparel_cost_index']]
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

### 해석 가이드

- **R² (Adj.)**: 0에 가까우면 매크로로 설명 안 됨 (=내부/공급사 요인 큼). 0.3 이상이면 매크로 영향 무시 못함.
- **계수의 부호**: 환율(원/달러) ↑ → 수입 단가 ↑ 가 정상. 부호가 반대면 데이터 점검.
- **p-value < 0.05**: 통계적으로 유의. 표본 36개월은 작은 편이라 결과는 보수적으로 해석.

## 5. 매크로 충격 시나리오 — 환율 +5% / 유가 +10%

회귀 계수를 단순 적용해 향후 12개월 매입가에 미칠 영향을 추정 (확정 예측이 아니라 시나리오 분석).

In [ ]:
scenarios = {
    'baseline':         {'fx': 0.00, 'oil': 0.00, 'app': 0.00},
    'fx +5%':           {'fx': 0.05, 'oil': 0.00, 'app': 0.00},
    'oil +10%':         {'fx': 0.00, 'oil': 0.10, 'app': 0.00},
    'fx +5% + oil +10%':{'fx': 0.05, 'oil': 0.10, 'app': 0.02},
}

current_cost = float(panel['unit_cost_krw'].iloc[-1])
params = model.params  # const, fx, oil, app
rows = []
for name, sh in scenarios.items():
    log_change = (
        params.get('fx_krw_usd', 0) * np.log(1 + sh['fx']) +
        params.get('oil_index_usd', 0) * np.log(1 + sh['oil']) +
        params.get('apparel_cost_index', 0) * np.log(1 + sh['app'])
    )
    new_cost = current_cost * np.exp(log_change)
    rows.append({'scenario': name, 'log_change': round(log_change, 4), 'new_cost_krw': round(new_cost, 0), 'pct_change': round((new_cost/current_cost - 1) * 100, 2)})
scen_df = pd.DataFrame(rows)
scen_df

## 6. 저장 — Claude Desktop이 사용할 매크로 인사이트

회귀 결과 + 시나리오 표를 markdown으로 export.

In [ ]:
md = []
md.append(f'# {TARGET} — Macro Context\n')
md.append(f'- 분석 기간: {panel.index.min().date()} ~ {panel.index.max().date()}\n')
md.append('## 동시 상관\n')
md.append(corr.to_frame('corr_with_unit_cost_krw').to_markdown())
md.append('\n## 시차 상관\n')
md.append(lag_df.to_markdown())
md.append('\n## OLS 회귀 (1차 로그 차분)\n')
md.append(f'- R² = {model.rsquared:.3f}, Adj. R² = {model.rsquared_adj:.3f}\n')
md.append('- 계수 (p-value):\n')
for name in model.params.index:
    md.append(f"  - {name}: {model.params[name]:+.3f}  (p={model.pvalues[name]:.3f})\n")
md.append('\n## 시나리오\n')
md.append(scen_df.to_markdown(index=False))
out_md = '\n'.join(md)
out_path = DATA_DIR / f'macro_context_{TARGET}.md'
out_path.write_text(out_md, encoding='utf-8')
print(f'Saved → {out_path.resolve()}')
print('\n', out_md[:1200])

## 다음 단계

1. `macro_context_*.md`을 Claude Desktop에 첨부 → `pattern_insights(si)` 또는 `procurement_strategy(si)` 호출.
2. 가격이 매크로 충격 후 평균 회귀하는 성격이라면 `06_price_signals.ipynb`로 매입 시그널 산출.
3. 매크로 변동성이 크다면 `07_derivatives_hedging.ipynb`로 헷징 시나리오 분석.